In [9]:
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from langgraph.constants import START, END
from langgraph.graph import MessagesState, StateGraph
from langgraph.checkpoint.memory import InMemorySaver

# 声明模型
load_dotenv(override=True)

model=ChatDeepSeek(
    model="deepseek-flash",
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)

# 声明状态
class OverAllState(MessagesState):
    output:str

# 声明节点
def llm_mode(state:OverAllState):
    messages=state["messages"]
    res=model.invoke(messages)
    return {
        "messages":[res]
    }

# 声明输出节点
def output_node(state:OverAllState)->OverAllState:
    return {
        "output":state["messages"][-1].content
    }


# 构建图
builder=StateGraph(state_schema=OverAllState)
builder.add_node("llm_node",llm_mode)
builder.add_node("output_node",output_node)

builder.add_edge(START,"llm_node")
builder.add_edge("llm_node","output_node")
builder.add_edge("output_node",END)

#配置检查点存储器
checkpointer =InMemorySaver()
graph=builder.compile(checkpointer=checkpointer)

# 使用的时候需要填写线程ID
config={
    "configurable":{
        "thread_id":"chapter03-1"
    }
}
# 执行
graph.invoke({"messages":[HumanMessage("你好，我是晴曛")]},config=config)


graph.invoke({"messages":[HumanMessage("你好，我是谁")]},config=config)

{'messages': [HumanMessage(content='你好，我是晴曛', additional_kwargs={}, response_metadata={}, id='27571c8a-a470-4923-85b8-6b418f358e96'),
  AIMessage(content='你好，晴曛！很高兴认识你。这个名字听起来很温暖，像午后阳光一样。有什么我可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 10, 'total_tokens': 36, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 10}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501', 'id': 'd911a022-dded-43ce-bac7-c7c8ec236b15', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1571c-e473-7453-93fb-0fa4114aa242-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 26, 'total_tokens': 36, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}),
  HumanMessage(content='你好，我是谁', additional_